# EEG Motor Imagery: Parallel CNN-RNN (Google Colab Version)

This notebook implements a Parallel CNN-RNN architecture for EEG motor imagery classification. It is designed to be self-contained and ready to run on Google Colab.

## 1. Setup Environment
First, we check for available hardware and install necessary dependencies.

In [36]:
# @title 1.1 Mount Google Drive (Optional)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted successfully.")
except ImportError:
    print("Not running on Google Colab. Skipping drive mount.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully.


In [37]:
# @title 1.2 Check Hardware and Install Dependencies
import torch
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    DEVICE = torch.device("cuda")
else:
    print("GPU not available, using CPU.")
    DEVICE = torch.device("cpu")

try:
    import seaborn as sns
    import sklearn
    import matplotlib
    print("Dependencies already installed.")
except ImportError:
    print("Installing dependencies...")
    install("seaborn")
    install("scikit-learn")
    install("matplotlib")

PyTorch version: 2.10.0+cu128
GPU available: Tesla T4
Dependencies already installed.


## 2. Configuration
Central configuration for hyperparameters and paths. **Note:** If you mounted your drive, update `PARALLEL_DATA_DIR` to point to your data folder (e.g., `/content/drive/MyDrive/...`).

In [44]:
import os

# ── Data paths ─────────────────────────────────────────────────────────────────
REPO_ROOT        = os.getcwd()
PARALLEL_DATA_DIR = "/content/drive/MyDrive/CascadeParallel/npz_4class_parallel_W10"
OUTPUT_DIR        = "/content/drive/MyDrive/CascadeParallel/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Shared hyperparameters ─────────────────────────────────────────────────────
WINDOW       = 10
N_CLASSES    = 4       # 4 motor-imagery classes
EPOCHS       = 20
SEED         = 42
TRAIN_RATIO  = 0.75
NUM_SUBJECTS = 10      # Limit to first N subjects (0 or None for all 108)

BATCH   = 32
LR      = 1e-4
DROPOUT = 0.5

# ── DataLoader settings ────────────────────────────────────────────────────────
NUM_WORKERS = 2
PREFETCH_FACTOR = 2
PIN_MEMORY  = DEVICE.type == "cuda"

# ── Parallel model config ──────────────────────────────────────────────────────
PARALLEL_CFG = dict(
    conv_channels=32,
    cnn_fc=256,
    n_electrodes=64,
    rnn_fc_in=256,
    lstm_hidden=16,
    lstm_layers=2,
    rnn_fc_out=256,
    fusion="concat",   # 'concat' | 'add' | 'concat_fc' | 'concat_conv1d'
)

## 3. Dataset Implementation
Memory-efficient lazy loading directly from the main `.npz` files using `mmap_mode` to keep memory usage low while avoiding chunking overhead.

In [45]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, random_split

class LazyEEGDataset(Dataset):
    def __init__(self, data_dir, num_subjects=None):
        labels_path = "/content/drive/MyDrive/CascadeParallel/S001_S108_win10_labels.npz"
        cnn_path = "/content/drive/MyDrive/CascadeParallel/S001_S108_win10_cnn_data.npz"
        rnn_path = "/content/drive/MyDrive/CascadeParallel/S001_S108_win10_rnn_data.npz"

        if not all(os.path.exists(p) for p in [labels_path, cnn_path, rnn_path]):
            print(os.path.exists(p) for p in [labels_path, cnn_path, rnn_path])
            print(f"WARNING: Data files not found in {data_dir}.")
            self.total_len = 0; self.classes = []
            return

        print(f"Loading labels...")
        labels_raw = np.load(labels_path, allow_pickle=True)["labels"]
        self.classes = sorted(list(set(labels_raw)))
        c2i = {c: i for i, c in enumerate(self.classes)}
        self.labels = np.array([c2i[l] for l in labels_raw], dtype=np.int64)
        
        print(f"Memory-mapping CNN & RNN data...")
        self.cnn_data = np.load(cnn_path, mmap_mode="r")["data"]
        self.rnn_data = np.load(rnn_path, mmap_mode="r")["data"]
        
        self.total_len = len(self.labels)
        if num_subjects and 0 < num_subjects < 108:
            self.total_len = int((num_subjects / 108) * self.total_len)

        print(f"Total samples = {self.total_len} | classes = {self.classes}")

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        if idx >= self.total_len: raise IndexError("Index out of bounds")
        
        # Slicing memmapped arrays is very fast
        cnn = self.cnn_data[idx].astype(np.float16)[:, np.newaxis, :, :]
        rnn = self.rnn_data[idx].astype(np.float16)
        label = self.labels[idx]
        
        return torch.from_numpy(cnn), torch.from_numpy(rnn), int(label)

def build_loaders(data_dir, num_subjects, train_ratio, batch_size, num_workers, pin_memory, prefetch_factor, seed):
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    dataset = LazyEEGDataset(data_dir, num_subjects=num_subjects)
    if len(dataset) == 0: return None, None, []

    n_train = int(train_ratio * len(dataset))
    n_test  = len(dataset) - n_train
    train_set, test_set = random_split(dataset, [n_train, n_test], generator=generator)

    _kwargs = dict(
        batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory,
        prefetch_factor=prefetch_factor, persistent_workers=(num_workers > 0),
    )
    train_loader = DataLoader(train_set, shuffle=True,  **_kwargs)
    test_loader  = DataLoader(test_set,  shuffle=False, **_kwargs)

    print(f"train={n_train:,} | test={n_test:,} | batches={len(train_loader)}")
    return train_loader, test_loader, dataset.classes

## 4. Model Definition
Parallel CNN-RNN architecture with multiple fusion strategies.

In [46]:
import torch.nn as nn

class ParallelCNNRNN(nn.Module):
    def __init__(self, window_size=10, conv_channels=32, cnn_fc=256, n_electrodes=64, 
                 rnn_fc_in=256, lstm_hidden=16, lstm_layers=2, rnn_fc_out=128, 
                 n_classes=4, dropout=0.5, fusion="concat", **kwargs):
        super().__init__()
        self.fusion = fusion
        ch = conv_channels

        self.cnn_features = nn.Sequential(
            nn.Conv2d(1, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ELU(inplace=True),
            nn.Conv2d(ch, ch*2, 3, padding=1), nn.BatchNorm2d(ch*2), nn.ELU(inplace=True),
            nn.Conv2d(ch*2, ch*4, 3, padding=1), nn.BatchNorm2d(ch*4), nn.ELU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.cnn_fc = nn.Sequential(
            nn.Flatten(), nn.Linear(ch * 4, cnn_fc), nn.ELU(inplace=True), nn.Dropout(dropout),
        )
        self.rnn_proj = nn.Sequential(
            nn.Linear(n_electrodes, rnn_fc_in), nn.ELU(inplace=True),
        )
        self.lstm = nn.LSTM(
            rnn_fc_in, lstm_hidden, lstm_layers, batch_first=True, 
            dropout=dropout if lstm_layers > 1 else 0.0
        )
        self.rnn_head = nn.Sequential(
            nn.Linear(lstm_hidden, rnn_fc_out), nn.ELU(inplace=True), nn.Dropout(dropout),
        )

        if fusion == "concat":
            fused_dim = cnn_fc + rnn_fc_out; self.fuse_layer = None
        elif fusion == "add":
            fused_dim = cnn_fc; self.fuse_layer = None
        elif fusion == "concat_fc":
            fused_dim = cnn_fc + rnn_fc_out
            self.fuse_layer = nn.Sequential(nn.Linear(fused_dim, fused_dim), nn.ELU(inplace=True))
        elif fusion == "concat_conv1d":
            fused_dim = cnn_fc + rnn_fc_out
            self.fuse_layer = nn.Sequential(nn.Conv1d(fused_dim, fused_dim, 1), nn.ELU(inplace=True))
        
        self.readout = nn.Linear(fused_dim, n_classes)

    def forward(self, cnn_x, rnn_x):
        B, W, C, H, Wd = cnn_x.shape
        cnn_frames = self.cnn_fc(self.cnn_features(cnn_x.reshape(B * W, C, H, Wd)))
        cnn_out = cnn_frames.reshape(B, W, -1).sum(dim=1)
        proj = self.rnn_proj(rnn_x.reshape(B * W, -1)).reshape(B, W, -1)
        lstm_h = self.lstm(proj)[0][:, -1, :]
        rnn_out = self.rnn_head(lstm_h)

        if self.fusion == "concat":
            fused = torch.cat([cnn_out, rnn_out], dim=1)
        elif self.fusion == "add":
            fused = cnn_out + rnn_out
        elif self.fusion == "concat_fc":
            fused = self.fuse_layer(torch.cat([cnn_out, rnn_out], dim=1))
        elif self.fusion == "concat_conv1d":
            cat = torch.cat([cnn_out, rnn_out], dim=1).unsqueeze(2)
            fused = self.fuse_layer(cat).squeeze(2)
        return self.readout(fused)

def build_model(cfg, window, n_classes, dropout):
    return ParallelCNNRNN(window_size=window, n_classes=n_classes, dropout=dropout, **cfg)

## 5. Trainer
Training and evaluation logic.

In [47]:
import time
from collections import defaultdict
from torch.amp import autocast, GradScaler

def train_epoch(model, loader, criterion, optimiser, scaler, device):
    model.train()
    total_loss, correct, n_samples = 0.0, 0, 0
    for cnn_x, rnn_x, labels in loader:
        cnn_x, rnn_x = cnn_x.to(device, dtype=torch.float32), rnn_x.to(device, dtype=torch.float32)
        labels = labels.to(device)
        optimiser.zero_grad(set_to_none=True)
        with autocast(device_type=device.type if device.type != 'mps' else 'cpu'):
            logits = model(cnn_x, rnn_x); loss = criterion(logits, labels)
        scaler.scale(loss).backward(); scaler.step(optimiser); scaler.update()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item(); n_samples += labels.size(0)
    return total_loss / n_samples, correct / n_samples

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n_samples = 0.0, 0, 0
    all_preds, all_labels = [], []
    for cnn_x, rnn_x, labels in loader:
        cnn_x, rnn_x = cnn_x.to(device, dtype=torch.float32), rnn_x.to(device, dtype=torch.float32)
        labels = labels.to(device)
        logits = model(cnn_x, rnn_x); loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1); correct += (preds == labels).sum().item()
        n_samples += labels.size(0); all_preds.extend(preds.cpu().tolist()); all_labels.extend(labels.cpu().tolist())
    return total_loss / n_samples, correct / n_samples, all_preds, all_labels

def run_training(model, train_loader, test_loader, device, epochs, lr, output_dir):
    criterion = nn.CrossEntropyLoss(); optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = GradScaler(enabled=(device.type == "cuda")); best_acc = 0.0; history = defaultdict(list)
    model.to(device)
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimiser, scaler, device)
        te_loss, te_acc, _, _ = evaluate(model, test_loader, criterion, device)
        history["tr_loss"].append(tr_loss); history["tr_acc"].append(tr_acc)
        history["te_loss"].append(te_loss); history["te_acc"].append(te_acc)
        if te_acc > best_acc:
            best_acc = te_acc; torch.save(model.state_dict(), os.path.join(output_dir, "best_parallel.pt"))
        if epoch % 5 == 0 or epoch == 1 or epoch == epochs:
            print(f"Ep {epoch:3d}/{epochs} | tr={tr_loss:.4f}/{tr_acc:.4f} | te={te_loss:.4f}/{te_acc:.4f} | best={best_acc:.4f} | {time.perf_counter()-t0:.1f}s")
    return dict(history), best_acc

## 6. Evaluation & Plotting
Visualizing the results.

In [48]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

def run_final_evaluation(model, test_loader, class_names, history, best_acc, device, output_dir):
    ckpt_path = os.path.join(output_dir, "best_parallel.pt")
    if os.path.exists(ckpt_path): model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device)
    
    _, final_acc, preds, true_labels = evaluate(model, test_loader, nn.CrossEntropyLoss(), device)
    print(f"\n🏆 Final test accuracy: {final_acc * 100:.2f}%")
    print("\nClassification Report:\n", classification_report(true_labels, preds, target_names=class_names))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(history['tr_loss'], label='Train Loss'); ax1.plot(history['te_loss'], label='Test Loss')
    ax1.set_title('Loss'); ax1.legend()
    ax2.plot(history['tr_acc'], label='Train Acc'); ax2.plot(history['te_acc'], label='Test Acc')
    ax2.set_title('Accuracy'); ax2.legend(); plt.show()

    cm = confusion_matrix(true_labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('True'); plt.show()

## 7. Execution
Run the training and evaluation loop.

In [49]:
# ── Build DataLoaders ──
train_loader, test_loader, class_names = build_loaders(
    data_dir=PARALLEL_DATA_DIR, num_subjects=NUM_SUBJECTS, train_ratio=TRAIN_RATIO,
    batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH_FACTOR, seed=SEED
)

if train_loader:
    model = build_model(PARALLEL_CFG, WINDOW, N_CLASSES, DROPOUT).to(DEVICE)
    history, best_acc = run_training(
        model=model, train_loader=train_loader, test_loader=test_loader,
        device=DEVICE, epochs=EPOCHS, lr=LR, output_dir=OUTPUT_DIR
    )
    run_final_evaluation(
        model=model, test_loader=test_loader, class_names=class_names,
        history=history, best_acc=best_acc, device=DEVICE, output_dir=OUTPUT_DIR
)

<generator object LazyEEGDataset.__init__.<locals>.<genexpr> at 0x7ce730378ee0>
